# Where may I fly, advanced: zone geometry, a map of allowed height, flight levels and the AIRAC cycle

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/eu_rules_pro_en.ipynb)

**OpenSkyLab · notebook for the lab** [Where and how may I fly](https://openskylab.space/edu_viz_eu_rules.html) · [česká verze](eu_rules_pro_cs.ipynb)

The advanced version of the notebook on UAS geographical zones. On current ŘLP data you find out which zones concern a drone at all according to their vertical limits, build a map of allowed flight height around Brno from the grids, compute from METAR reports how high flight levels really are, check the effective dates of zones against the AIRAC cycle and draw a vertical cross-section of the zones from Bystrc to Tuřany airport over the real terrain. This is a teaching aid, not legal advice: before a flight always check the zones in the DroneMap app and the conditions in the CAA measures of general nature (OOP).

**Goal:** sort zones by their lower limit and determine where they concern a drone, combine the GRID_CTR and GRID_ATZ grids into a map of allowed height, convert flight levels to true altitude from QNH and temperature, check effective dates against the AIRAC cycle and build a vertical cross-section of the zones over the DMR 5G terrain.

**Data:** UAS geographical zones as GeoJSON (reference boundaries) from the ŘLP page https://aim.rlp.cz/?lang=en&p=uas-gz, downloaded at run time, except layers larger than the chosen limit; METAR reports of Brno-Tuřany airport (LKTB) for the last 7 days from the Aviation Weather Center (NOAA); terrain elevation from the Digital terrain model of the Czech Republic, 5th generation (DMR 5G, ČÚZK). The conversion of a flight level to true altitude uses the ICAO standard atmosphere with the temperature adjusted to the airport; it is an approximation.

**Tasks:** five. Write your answers in the *Your answer* cells below the tasks. Lines marked `# ◀ parameter` are meant to be changed; run the cell again with Shift + Enter.

**Time:** about 60 min. You need a Google account (Colab); nothing is installed on your computer. Run the cells in order from top to bottom.

> **💻 What the code does:** Optional cell. If you paste the settings copied from the lab and run it, the cells below take the same parameter values and say so for each one. If you leave it empty, the default values are used. It downloads nothing.

In [ ]:
# @title Settings from the lab (optional)
# @markdown In the lab, click the Notebook link. The settings are copied; paste them here in place of `{}` (Ctrl+V) and run the cell.
Z_VIZUALU = {}  # @param {type:"raw"}
STEM = "eu_rules"


def z_viz(name, default, mapping=None):
    """Value of a parameter from the lab if the pasted code contains it; otherwise the default value."""
    z = (
        Z_VIZUALU
        if isinstance(Z_VIZUALU, dict) and Z_VIZUALU.get("osl") == STEM
        else {}
    )
    if name not in z:
        return default
    v = z[name]
    if mapping is not None:
        if v not in mapping:
            print(
                f"{name}: value {v!r} from the lab is not in this notebook, keeping {default!r}"
            )
            return default
        v = mapping[v]
    elif isinstance(default, bool):
        v = bool(v)
    elif isinstance(default, int):
        v = int(round(v))
    elif isinstance(default, float):
        v = float(v)
    print(f"{name} = {v!r}  (from the lab)")
    return v


if isinstance(Z_VIZUALU, dict) and Z_VIZUALU and Z_VIZUALU.get("osl") != STEM:
    print(
        f"The pasted code is from another lab ({Z_VIZUALU.get('osl')}), the notebook does not use it."
    )

## 1. Setup and zones

We load the list of current zones from the ŘLP page and download all layers up to the chosen size limit and both grids. Downloading takes about a minute.

> **💻 What the code does:** Installs the libraries shapely (geometry), pyproj (coordinate transformations) and rasterio (rasters).

In [ ]:
%pip install -q shapely pyproj rasterio

> **💻 What the code does:** Loads the libraries and defines `cuzk_raster`, which downloads a window of a ČÚZK raster (DMR 5G terrain or DMP 1G surface model) as an array of elevations. Nothing is downloaded yet.

In [ ]:
import html
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
import shapely
from pyproj import Transformer
from shapely.geometry import LineString, box, shape


def cuzk_raster(service, x0, y0, x1, y1, res=2.0):
    """Window of a ČÚZK raster (dmr5g, dmp1g, dmp_obrazova_korelace) in S-JTSK (EPSG:5514), heights in Bpv.
    Returns the height array (row 0 = north) and its transform."""
    nx, ny = int(round((x1 - x0) / res)), int(round((y1 - y0) / res))
    url = f"https://ags.cuzk.cz/arcgis2/rest/services/{service}/ImageServer/exportImage"
    params = {
        "bbox": f"{x0},{y0},{x1},{y1}",
        "bboxSR": 5514,
        "imageSR": 5514,
        "size": f"{nx},{ny}",
        "format": "tiff",
        "pixelType": "F32",
        "interpolation": "RSP_BilinearInterpolation",
        "f": "image",
    }
    for attempt in range(4):
        r = requests.get(url, params=params, timeout=120)
        if r.ok and r.headers.get("content-type", "").startswith("image"):
            break
        time.sleep(3 * (attempt + 1))
    r.raise_for_status()
    with rasterio.MemoryFile(r.content) as mem, mem.open() as ds:
        return ds.read(1).astype(float), ds.transform

> **💻 What the code does:** Downloads the list of current UAS geographical zones from the ŘLP AIM page and then all layers up to `MAX_MB` and always both grids (GRID_CTR, GRID_ATZ). For each feature it keeps the geometry, identifier, vertical limits and effective date. It takes about a minute.

In [ ]:
AIM = "https://aim.rlp.cz"


def get(url, tries=4, **kw):
    for i in range(tries):
        try:
            r = requests.get(url, timeout=300, **kw)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            print(url.split("/")[-1], type(e).__name__, "– retrying")
            time.sleep(5 * (i + 1))
    raise RuntimeError(f"Could not download {url}")


page = get(f"{AIM}/?lang=en&p=uas-gz").content.decode("utf-8")
rows = []
for tr in re.findall(r"<tr.*?</tr>", page, re.S):
    href = re.search(r'href="(/data/uas/[^"]+\.json)"', tr)
    cols = [
        re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", c))).strip()
        for c in re.findall(r"<td[^>]*>(.*?)</td>", tr, re.S)
    ]
    if href and len(cols) >= 4 and "/actual/" in href.group(1):
        mb = re.search(r"([\d.,]+)\s*(kB|MB)", cols[3])
        rows.append(
            {
                "code": cols[2].replace(".json", ""),
                "desc": cols[1],
                "MB": float(mb.group(1).replace(",", "."))
                * (1e-3 if mb.group(2) == "kB" else 1),
                "url": AIM + href.group(1),
            }
        )
zones = pd.DataFrame(rows).set_index("code")
# Largest file to download (MB); both grids are always downloaded.
MAX_MB = 20  # ◀ parameter
T_UTM = Transformer.from_crs(4326, 32633, always_xy=True)


def to_utm(g):
    return shapely.transform(
        g, lambda xy: np.column_stack(T_UTM.transform(xy[:, 0], xy[:, 1]))
    )


layers = {}
for code, z in zones.iterrows():
    if z.MB <= MAX_MB or code in ("GRID_CTR", "GRID_ATZ"):
        feats = [f for f in get(z.url).json()["features"] if f.get("geometry")]
        layers[code] = pd.DataFrame(
            {
                "geom": [shape(f["geometry"]) for f in feats],
                **{
                    k: [f["properties"].get(k) for f in feats]
                    for k in ("ident", "vertical_limit", "effective_date")
                },
            }
        )
print(
    f"downloaded {len(layers)} layers, {sum(len(v) for v in layers.values())} features; large layers left out:",
    ", ".join(c for c in zones.index if c not in layers),
)

## 2. Which zones concern a drone at all?

Every zone applies only between its lower and upper limit. The lower limit can be at the ground (GND), a height above ground (AGL), an altitude above sea level (AMSL) or a flight level (FL, here converted with the standard atmosphere). For limits in AMSL and FL the terrain matters: we decide whether the zone reaches a drone at the chosen height everywhere in Czechia, nowhere, or only above a certain terrain altitude.

> **💻 What the code does:** For each combination of layer and vertical limits it converts the lower limit to an altitude above sea level (or keeps it at ground level or above ground) and compares it with the flight height `DRONE_AGL` above the lowest and highest point of Czechia. It prints whether the zone concerns the drone everywhere, nowhere or only above a certain terrain elevation, and how many features start at the ground and how many higher up.

In [ ]:
# Lowest and highest point of Czechia (m a.s.l.): the Elbe at Hřensko and Sněžka.
H_MIN, H_MAX = 115, 1603
DRONE_AGL = 120  # ◀ parameter: flight height above ground (m)


def lower_amsl(txt):
    """Lower limit of a zone as altitude (m); returns the type of limit and the value."""
    s = txt.split("-", 1)[0].strip().upper().replace(" ", "")
    if s in ("GND", "SFC"):
        return "GND", 0.0
    if s.startswith("FL"):
        return "FL", float(s[2:]) * 100 * 0.3048
    m = re.match(r"([\d.]+)(FT|M)(AGL|AMSL|MSL)", s)
    v = float(m.group(1)) * (0.3048 if m.group(2) == "FT" else 1)
    return ("AGL", v) if m.group(3) == "AGL" else ("AMSL", v)


rows = []
for code, df in layers.items():
    if code.startswith("GRID"):
        continue
    for vl, n in df.vertical_limit.value_counts().items():
        kind, v = lower_amsl(vl)
        if kind in ("GND", "AGL"):
            where = "everywhere" if v <= DRONE_AGL else "nowhere"
        else:  # AMSL or FL: depends on the terrain
            where = (
                "nowhere"
                if v > H_MAX + DRONE_AGL
                else (
                    "everywhere"
                    if v <= H_MIN + DRONE_AGL
                    else f"where the terrain is above {v - DRONE_AGL:.0f} m a.s.l."
                )
            )
        rows.append(
            {
                "layer": code,
                "vertical limits": vl,
                "features": n,
                "lower limit": kind,
                "lower limit (m a.s.l.)": None if kind in ("GND", "AGL") else round(v),
                f"affects a drone at {DRONE_AGL} m": where,
            }
        )
lim = pd.DataFrame(rows)
pd.set_option("display.max_rows", 100)
print(lim.to_string(index=False))
print(lim.groupby("lower limit")["features"].sum().to_string())

> **✏️ Task 1.** How many features start at the ground and how many only at height? Which layers start high, and why do they not concern a drone at 120 m in the lowlands? Find the zones that reach a drone only in the mountains and determine above what terrain altitude. Set DRONE_AGL to 50 m and 150 m: what changes? Why is it not enough to know only the horizontal boundary of a zone?

*Your answer:* 

## 3. A map of allowed height around Brno

The GRID_CTR grid (around controlled airports, 30″ cells) and the GRID_ATZ grid (areas of uncontrolled air traffic, 15″ cells) state in every cell up to what height above ground flight is allowed without further approval. We sample the area every 7.5″, so each point lies inside one cell. Besides that we count how many zones from the ground (GND) overlap at a point.

> **💻 What the code does:** Samples the area `BBOX` every 7.5″ and at each point finds the permitted height above ground from both grids and the number of overlapping zones starting at the ground. It prints the share of the area by permitted height and plots both maps.

In [ ]:
# Longitude and latitude: west, south, east, north.
BBOX = (16.45, 49.10, 16.80, 49.30)  # ◀ parameter: area
STEP = 1 / 480  # 7.5″ step: half a GRID_ATZ cell, a quarter of a GRID_CTR cell
lon = np.arange(BBOX[0] + STEP / 2, BBOX[2], STEP)
lat = np.arange(BBOX[1] + STEP / 2, BBOX[3], STEP)
LON, LAT = np.meshgrid(lon, lat)


def hmax(txt):
    return float(re.search(r"-\s*([\d.]+)\s*m\s*AGL", txt).group(1))


allowed = np.full(LON.shape, np.nan)
for code in ("GRID_CTR", "GRID_ATZ"):
    for g, v in zip(layers[code].geom, layers[code].vertical_limit):
        if g.intersects(box(*BBOX)):
            inside = shapely.contains_xy(g, LON, LAT)
            allowed[inside] = np.fmin(allowed[inside], hmax(v))
# Zones from the ground (GND).
restr = [
    c
    for c in layers
    if c.startswith(("LKR312", "LKR313", "LKR314", "LKR315", "LKR318", "LKR319"))
]
nzones = np.zeros(LON.shape, int)
for code in restr:
    for g in layers[code].geom:
        if g.intersects(box(*BBOX)):
            nzones += shapely.contains_xy(g, LON, LAT)
classes = pd.cut(
    pd.Series(allowed.ravel()),
    [-1, 0, 30, 60, 90, 119, 120],
    labels=["0 m", "1–30 m", "31–60 m", "61–90 m", "91–119 m", "120 m"],
)
share = (100 * classes.value_counts(normalize=False) / allowed.size).reindex(
    classes.cat.categories
)
share["outside the grid"] = 100 * np.isnan(allowed).mean()
print("share of the area by allowed height from GRID_CTR and GRID_ATZ (%):")
print(share.round(1).to_string())
print(
    f"share of the area in at least one zone from the ground (layers {', '.join(restr)}): {100 * (nzones > 0).mean():.0f} %"
)
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
ext = (BBOX[0], BBOX[2], BBOX[1], BBOX[3])
asp = 1 / np.cos(np.radians(LAT.mean()))
im = ax[0].imshow(
    allowed, origin="lower", extent=ext, cmap="RdYlGn", vmin=0, vmax=120, aspect=asp
)
plt.colorbar(im, ax=ax[0], label="allowed height (m AGL)")
ax[0].set_title("allowed height from the grids (grey = outside the grid)")
ax[0].set_facecolor("0.8")
im = ax[1].imshow(nzones, origin="lower", extent=ext, cmap="viridis", aspect=asp)
plt.colorbar(im, ax=ax[1], label="number of zones")
ax[1].set_title("number of zones from the ground overlapping at a point")
plt.show()

> **✏️ Task 2.** What share of the area may you fly over at 120 m according to the grids, what share only up to 30 m, and where may you not fly at all? Where is the allowed height 0 m, and why there (compare with the position of Tuřany airport, Medlánky airfield and the heliports in the right map)? Does the grey area outside the grid mean you may fly there without restrictions? Remember the large layers we did not download.

*Your answer:* 

## 4. How high is a flight level?

A flight level FL is a pressure altitude in hundreds of feet at the standard pressure of 1013.25 hPa. The true altitude of the level depends on the pressure at the ground (QNH) and on the air temperature: in warm air it lies higher, in cold air lower. We take QNH and temperature from METAR reports for the last 7 days and compute the true altitude of the levels that are the lower limit of LKR320A areas.

> **💻 What the code does:** Downloads the METAR reports of the last 7 days from airport `STATION` and from QNH and temperature computes the true altitude of the flight levels `FLS` for each report. It plots how it varies and prints the minimum, maximum and range.

In [ ]:
# METAR reports.
STATION = "LKTB"  # ◀ parameter: the airport whose QNH and temperature we use
FLS = [65, 95]  # ◀ parameter: flight levels (e.g. lower limits of LKR320A areas)
met = pd.DataFrame(
    get(
        "https://aviationweather.gov/api/data/metar",
        params={"ids": STATION, "format": "json", "hours": 168},
    ).json()
)
met["time"] = pd.to_datetime(met.reportTime)
met = met.dropna(subset=["altim", "temp"]).sort_values("time").set_index("time")
h_s = float(met.elev.iloc[0])


# pressure altitude (m) in ISA
def pa(p):
    return 44330.77 * (1 - (p / 1013.25) ** 0.190263)


def true_alt(fl, qnh, t_s):
    """True altitude of a flight level: first the altitude on QNH, then a correction for the temperature of the layer between the airport and the level."""
    z_q = fl * 100 * 0.3048 - pa(qnh)  # altitude on an altimeter set to QNH
    t_isa = 288.15 - 0.0065 * (h_s + z_q) / 2  # mean temperature of the layer in ISA
    # Mean actual temperature of the layer (lapse 6.5 K/km).
    t_act = t_s + 273.15 - 0.0065 * (z_q - h_s) / 2
    return h_s + (z_q - h_s) * t_act / t_isa


fig, ax = plt.subplots(figsize=(12, 4))
rows = {}
for fl in FLS:
    z = true_alt(fl, met.altim, met.temp)
    ax.plot(z.index, z, label=f"FL{fl}")
    ax.axhline(fl * 100 * 0.3048, color="k", lw=0.8, ls=":")
    rows[f"FL{fl}"] = {
        "FL × 100 ft (m)": fl * 100 * 0.3048,
        "min (m a.s.l.)": z.min(),
        "max (m a.s.l.)": z.max(),
        "range (m)": z.max() - z.min(),
    }
print(
    f"{STATION}: {len(met)} METAR reports, QNH {met.altim.min():.0f}–{met.altim.max():.0f} hPa, temperature {met.temp.min():.0f}–{met.temp.max():.0f} °C, airport {h_s:.0f} m a.s.l."
)
print(pd.DataFrame(rows).T.to_string(float_format=lambda v: f"{v:.0f}"))
ax.set_ylabel("true altitude (m a.s.l.)")
ax.legend()
ax.grid(alpha=0.3)
ax.set_title(f"true altitude of flight levels from QNH and temperature at {STATION}")
plt.show()

> **✏️ Task 3.** By how many metres did the true altitude of FL65 and FL95 change during the week? What affects it more, QNH or temperature? Compute how many metres of height correspond to a change of QNH by 1 hPa. How high above Děvín (about 550 m a.s.l.) does the LKTB4 area (FL65) start in the lowest and the highest case? Why are higher limits in aviation given as flight levels, although their true altitude varies?

*Your answer:* 

## 5. When zones apply: the AIRAC cycle

Changes to aeronautical information are published for the fixed dates of the AIRAC cycle, which follow every 28 days, and ŘLP updates the zone data in line with this cycle. Every zone feature has an effective date (effective_date). We find out which effective dates are AIRAC dates.

> **💻 What the code does:** Groups all downloaded features by effective date and for each date determines whether it is an AIRAC cycle date (28 days from the reference date) and, if so, its number. It prints the table and the next AIRAC date.

In [ ]:
# A known AIRAC date (cycle 2401), cycles follow every 28 days.
AIRAC_REF = pd.Timestamp("2024-01-25", tz="UTC")
eff = pd.concat(
    [
        pd.DataFrame({"layer": c, "effective": pd.to_datetime(d.effective_date)})
        for c, d in layers.items()
    ]
)
tab = eff.groupby("effective")["layer"].agg(
    ["count", lambda s: ", ".join(sorted(set(s)))]
)
tab.columns = ["features", "layers"]
days = (tab.index - AIRAC_REF).days
tab["AIRAC date"] = days % 28 == 0
tab["cycle"] = [
    f"{(AIRAC_REF + pd.Timedelta(days=28 * (d // 28))).year % 100:02d}{1 + ((AIRAC_REF + pd.Timedelta(days=28 * (d // 28))) - pd.Timestamp(f'{(AIRAC_REF + pd.Timedelta(days=28 * (d // 28))).year}-01-01', tz='UTC')).days // 28:02d}"
    if d % 28 == 0
    else ""
    for d in days
]
pd.set_option("display.max_colwidth", 80)
print(tab.to_string())
today = pd.Timestamp.now(tz="UTC").normalize()
nxt = AIRAC_REF + pd.Timedelta(days=28 * ((today - AIRAC_REF).days // 28 + 1))
print(f"today {today:%Y-%m-%d}, next AIRAC date {nxt:%Y-%m-%d}")

> **✏️ Task 4.** Which effective dates match the AIRAC cycle and which do not? Which layers changed most often, and why those? The OOPUAS-0055 zone has a date outside the cycle: find on the ŘLP page or in DroneMap what zone it is, and explain why it can be created outside the cycle. When is the next AIRAC date, and what do you do before a flight after that date?

*Your answer:* 

## 6. A vertical cross-section of the zones

Along the line from Bystrc to Brno-Tuřany airport we sample the terrain from DMR 5G every 25 m, and for every zone the line crosses we convert the lower and upper limit to altitude (AGL above the terrain, FL as in the standard atmosphere). The black line is the terrain plus the allowed height from the grids.

> **💻 What the code does:** Along the line from `A` to `B` it samples the DMR 5G terrain every 25 m. For each zone the line crosses it plots the lower and upper limit as altitude, together with the terrain, the permitted height from the grids and the flight height `FLIGHT_AGL` (it can come from the lab). It prints on what share of the profile a flight at that height is possible according to the grids.

In [ ]:
A = (49.2467, 16.4798)  # ◀ parameter: start of the profile (Bystrc, Kozí horka)
B = (49.1510, 16.6940)  # ◀ parameter: end of the profile (Brno-Tuřany airport)
# M.
FLIGHT_AGL = z_viz("FLIGHT_AGL", 120)  # ◀ parameter: planned flight height above ground
T_JTSK = Transformer.from_crs(4326, 5514, always_xy=True)
(xa, xb), (ya, yb) = T_JTSK.transform([A[1], B[1]], [A[0], B[0]])
dem, tr = cuzk_raster(
    "dmr5g",
    min(xa, xb) - 100,
    min(ya, yb) - 100,
    max(xa, xb) + 100,
    max(ya, yb) + 100,
    res=10.0,
)
n = int(np.hypot(xb - xa, yb - ya) // 25) + 1
d = np.linspace(0, np.hypot(xb - xa, yb - ya), n)
xs = xa + (xb - xa) * d / d[-1]
ys = ya + (yb - ya) * d / d[-1]
col, row = ~tr * (xs, ys)
terr = dem[
    np.clip(row.astype(int), 0, dem.shape[0] - 1),
    np.clip(col.astype(int), 0, dem.shape[1] - 1),
]
# Over 15 km a linear interpolation in geographic coordinates is enough.
plon = np.linspace(A[1], B[1], n)
plat = np.linspace(A[0], B[0], n)


def amsl(txt, ground):
    s = txt.strip().upper().replace(" ", "")
    if s in ("GND", "SFC"):
        return ground
    if s.startswith("FL"):
        return np.full_like(ground, float(s[2:]) * 100 * 0.3048)
    m = re.match(r"([\d.]+)(FT|M)(AGL|AMSL|MSL)", s)
    v = float(m.group(1)) * (0.3048 if m.group(2) == "FT" else 1)
    return ground + v if m.group(3) == "AGL" else np.full_like(ground, v)


line = LineString(zip(plon, plat))
fig, ax = plt.subplots(figsize=(15, 6))
ax.fill_between(d / 1000, 0, terr, color="0.55", zorder=3)
k = 0
for code, df in layers.items():
    if code.startswith("GRID"):
        continue
    for g, ident, vl in zip(df.geom, df.ident, df.vertical_limit):
        if not g.intersects(line):
            continue
        inside = shapely.contains_xy(g, plon, plat)
        lo, hi = vl.split("-", 1)
        ax.fill_between(
            d / 1000,
            amsl(lo, terr),
            amsl(hi, terr),
            where=inside,
            alpha=0.25,
            color=f"C{k % 10}",
            label=f"{code} {ident}: {vl}",
        )
        k += 1
grid_h = np.full(n, np.nan)
for code in ("GRID_CTR", "GRID_ATZ"):
    for g, v in zip(layers[code].geom, layers[code].vertical_limit):
        if g.intersects(line):
            inside = shapely.contains_xy(g, plon, plat)
            grid_h[inside] = np.fmin(grid_h[inside], hmax(v))
ax.plot(
    d / 1000,
    terr + grid_h,
    "k",
    lw=2,
    zorder=4,
    label="terrain + allowed height from the grid",
)
ax.plot(
    d / 1000,
    terr + FLIGHT_AGL,
    "b--",
    zorder=4,
    label=f"flight {FLIGHT_AGL} m above ground",
)
ax.set_ylim(0, 1300)
ax.set_xlim(0, d[-1] / 1000)
ax.set_xlabel("distance from the start of the profile (km)")
ax.set_ylabel("altitude (m a.s.l.)")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
fig.subplots_adjust(right=0.72)
ax.set_title(
    "vertical cross-section of the zones: lower and upper limits converted to altitude (FL taken as in the standard atmosphere)"
)
plt.show()
ok = np.nan_to_num(grid_h, nan=120) >= FLIGHT_AGL
print(
    f"a flight at {FLIGHT_AGL} m above ground is possible according to the grids along {100 * ok.mean():.0f} % of the profile; terrain {terr.min():.0f}–{terr.max():.0f} m a.s.l."
)

> **✏️ Task 5.** Along what percentage of the profile may you fly at 120 m above ground according to the grids, and where does the allowed height drop to zero? Which zones does the profile cross from the ground and which start high above the drone? Set FLIGHT_AGL to 50 m: how does the result change? Why is it not enough for a drone flight to know only the zones the take-off site falls into?

*Your answer:* 

> **✏️ Extension (optional).** Set BBOX, A, B and STATION to the area around Václav Havel Airport (STATION = "LKPR", end of the profile 50.10° N, 14.26° E) and compare the map of allowed height and the cross-section with Brno.

*Your answer:* 

---
### Data sources and references

- UAS geographical zones: Air Navigation Services of the Czech Republic (ŘLP ČR), AIM, https://aim.rlp.cz/?lang=en&p=uas-gz; zone map: https://dronemap.gov.cz
- Commission Implementing Regulation (EU) 2019/947 on the rules and procedures for the operation of unmanned aircraft, Article 15 (UAS geographical zones).
- Civil Aviation Authority of the Czech Republic: measures of general nature on UAS geographical zones (LKR311 to LKR320), https://www.caa.gov.cz
- METAR reports: NOAA Aviation Weather Center, Data API, https://aviationweather.gov/data/api/
- Terrain elevation: Digital terrain model of the Czech Republic, 5th generation (DMR 5G), © ČÚZK, licence CC BY 4.0, service https://ags.cuzk.cz/arcgis2/rest/services/dmr5g/ImageServer
- ICAO (1993). *Manual of the ICAO Standard Atmosphere*, Doc 7488, 3rd ed. (pressure altitude).
- ICAO, Annex 15 to the Convention on International Civil Aviation, *Aeronautical Information Services* (the AIRAC system, 28-day cycle).

### How to cite

OpenSkyLab (2026). Where may I fly, advanced: zone geometry, a map of allowed height, flight levels and the AIRAC cycle [notebook, teaching resource], version 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/eu_rules_pro_en.ipynb. Licence CC BY 4.0.

Notebook text and code: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data are under their providers' licences (see above).

*This resource was developed by the OpenSkyLab team with the assistance of the generative AI Claude (Anthropic), used for drafting code and text. The authors are responsible for the content.*